# Generate TRAPPIST-1 SPHINX Contamination Curves

This notebook generates the `sphinx_TRAPPIST-1_contam_*` files from the local SPHINX stellar grids in `sphinx_data/`. The output curves use the same spot and facula coverage fractions as the PHOENIX curves used by the Earth-like retrieval campaign.


## Setup

Run this notebook from `Earth_like_Atmosphere/stellar_contamination`. The constants below define the TRAPPIST-1 photosphere, the spot/facula temperatures, and the coverage fractions used in the current comparison.


In [ ]:
from __future__ import annotations

import itertools
import re
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
SPHINX_DATA_DIR = ROOT / "sphinx_data"
WAVELENGTH_GRID_PATH = ROOT.parent / "waves.txt"

# TRAPPIST-1 stellar parameters for the quiet photosphere.
T_S = 2566.0
LOG_G_S = 5.25
MET_S = 0.0
C_TO_O = 0.5

# Heterogeneous-surface temperatures matched to the PHOENIX setup.
T_SPOT = 0.86 * T_S
T_FAC = T_S + 100.0

# Spot and facula coverage fractions used in the campaign grid.
F_SPOT_VALUES = np.array([0.01, 0.08, 0.26], dtype=float)
F_FAC_VALUES = np.array([0.08, 0.54, 0.70], dtype=float)

SPHINX_NAME = re.compile(
    r"Teff_(?P<T>[0-9]+(?:\.[0-9]+)?)_"
    r"logg_(?P<g>[0-9]+(?:\.[0-9]+)?)_"
    r"logZ_(?P<Z>[+\-]?[0-9]+(?:\.[0-9]+)?)_"
    r"CtoO_(?P<C>[0-9]+(?:\.[0-9]+)?)",
    re.IGNORECASE,
)


## Read the Local SPHINX Grid

The SPHINX filenames encode the stellar parameters. These helpers build a simple lookup table and load spectra onto the wavelength grid used by the campaign.


In [ ]:
def build_sphinx_index(dirpath: Path) -> dict[tuple[float, float, float, float], Path]:
    """Map each available `(Teff, logg, logZ, C/O)` tuple to its spectrum file."""
    index = {}

    for path in sorted(dirpath.glob("Teff_*")):
        match = SPHINX_NAME.search(path.name)
        if match is None:
            continue

        key = (
            float(match.group("T")),
            float(match.group("g")),
            float(match.group("Z")),
            float(match.group("C")),
        )
        index[key] = path

    if not index:
        raise RuntimeError(f"No SPHINX spectra were found in {dirpath}.")

    return index


def load_wavelength_grid(path: Path) -> np.ndarray:
    """Load the one-dimensional campaign wavelength grid in microns."""
    wl_um = np.loadtxt(path, dtype=float)
    if wl_um.ndim != 1:
        raise ValueError(f"The wavelength grid in {path} must be one-dimensional.")
    return wl_um


def load_sphinx_spectrum(path: Path) -> tuple[np.ndarray, np.ndarray]:
    """Load one SPHINX spectrum and return sorted wavelength and flux arrays."""
    data = np.loadtxt(path, dtype=float, comments="#")
    wl = data[:, 0]
    flux = data[:, 1]

    valid = np.isfinite(wl) & np.isfinite(flux) & (flux > 0.0)
    order = np.argsort(wl[valid])
    return wl[valid][order], flux[valid][order]


def regrid_spectrum(path: Path, wl_um: np.ndarray) -> np.ndarray:
    """Interpolate a SPHINX spectrum onto the campaign wavelength grid."""
    wl_src, flux_src = load_sphinx_spectrum(path)
    return np.interp(wl_um, wl_src, flux_src, left=flux_src[0], right=flux_src[-1])


## Interpolate in Stellar Temperature

The local SPHINX grid has discrete effective temperatures. For TRAPPIST-1, the photosphere, spot, and facula temperatures are evaluated by exact lookup when available, or by linear interpolation between the two neighboring temperature nodes.


In [ ]:
def available_temperatures(
    idx: dict[tuple[float, float, float, float], Path],
    logg: float,
    metallicity: float,
    c_to_o: float,
) -> np.ndarray:
    """Return the SPHINX temperatures available for a fixed stellar-parameter slice."""
    temps = [
        temp
        for temp, g_val, z_val, c_val in idx
        if (g_val, z_val, c_val) == (logg, metallicity, c_to_o)
    ]
    return np.array(sorted(temps), dtype=float)


def spectrum_at_temperature(
    idx: dict[tuple[float, float, float, float], Path],
    teff: float,
    logg: float,
    metallicity: float,
    c_to_o: float,
    wl_um: np.ndarray,
) -> np.ndarray:
    """Return a SPHINX spectrum at `teff`, interpolating in temperature if needed."""
    temps = available_temperatures(idx, logg, metallicity, c_to_o)
    if temps.size == 0:
        raise RuntimeError(
            f"No SPHINX spectra match logg={logg:.2f}, logZ={metallicity:+.1f}, C/O={c_to_o:.2f}."
        )

    exact_key = (teff, logg, metallicity, c_to_o)
    if exact_key in idx:
        return regrid_spectrum(idx[exact_key], wl_um)

    if teff < temps[0] or teff > temps[-1]:
        raise RuntimeError(
            f"Teff={teff:.2f} K is outside the available SPHINX range "
            f"({temps[0]:.1f}-{temps[-1]:.1f} K)."
        )

    upper = int(np.searchsorted(temps, teff, side="right"))
    t_lo = float(temps[upper - 1])
    t_hi = float(temps[upper])

    flux_lo = regrid_spectrum(idx[(t_lo, logg, metallicity, c_to_o)], wl_um)
    flux_hi = regrid_spectrum(idx[(t_hi, logg, metallicity, c_to_o)], wl_um)

    weight = (teff - t_lo) / (t_hi - t_lo)
    return (1.0 - weight) * flux_lo + weight * flux_hi


## Compute Contamination Curves

For each coverage pair, the quiet photosphere, spot, and facula spectra are combined into a disk-integrated stellar spectrum. The saved contamination factor is `epsilon(lambda) = F_phot(lambda) / F_disk(lambda)`.


In [ ]:
def compute_epsilon(
    wl_um: np.ndarray,
    idx: dict[tuple[float, float, float, float], Path],
    f_spot: float,
    f_fac: float,
) -> np.ndarray:
    """Compute `epsilon(lambda)` for one spot/facula coverage pair."""
    f_quiet = 1.0 - f_spot - f_fac
    if f_quiet <= 0.0:
        raise ValueError(
            f"Invalid coverage fractions: f_spot={f_spot:.2f}, f_fac={f_fac:.2f}; "
            "their sum must be smaller than one."
        )

    f_phot = spectrum_at_temperature(idx, T_S, LOG_G_S, MET_S, C_TO_O, wl_um)
    f_spot_spec = spectrum_at_temperature(idx, T_SPOT, LOG_G_S, MET_S, C_TO_O, wl_um)
    f_fac_spec = spectrum_at_temperature(idx, T_FAC, LOG_G_S, MET_S, C_TO_O, wl_um)

    f_disk = (f_quiet * f_phot) + (f_spot * f_spot_spec) + (f_fac * f_fac_spec)
    return f_phot / f_disk


def save_curve(wl_um: np.ndarray, epsilon: np.ndarray, f_spot: float, f_fac: float) -> Path:
    """Save one SPHINX contamination curve using the campaign naming convention."""
    output_path = ROOT / f"sphinx_TRAPPIST-1_contam_fspot{f_spot:.2f}_ffac{f_fac:.2f}.txt"
    np.savetxt(output_path, np.column_stack((wl_um, epsilon)))
    return output_path


## Run the Generation

This cell evaluates all nine coverage combinations and writes the output files in the current directory.


In [ ]:
idx = build_sphinx_index(SPHINX_DATA_DIR)
wl_um = load_wavelength_grid(WAVELENGTH_GRID_PATH)

generated_files = []
for f_spot, f_fac in itertools.product(F_SPOT_VALUES, F_FAC_VALUES):
    epsilon = compute_epsilon(wl_um, idx, float(f_spot), float(f_fac))
    output_path = save_curve(wl_um, epsilon, float(f_spot), float(f_fac))
    generated_files.append(output_path.name)
    print(f"[SPHINX] Saved {output_path.name}")

generated_files


## Plot the Generated Epsilon Curves

This final cell reloads the generated files and plots the contamination factors for quick visual inspection.


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(11, 7))

for filename in generated_files:
    path = ROOT / filename
    data = np.loadtxt(path)
    label = filename.replace("sphinx_TRAPPIST-1_contam_", "").replace(".txt", "")
    plt.plot(data[:, 0], data[:, 1], linewidth=1.6, label=label)

plt.axhline(1.0, color="0.5", linestyle="--", linewidth=1.0, alpha=0.7)
plt.xlabel("Wavelength [um]")
plt.ylabel("epsilon(lambda)")
plt.title("SPHINX stellar contamination curves for TRAPPIST-1")
plt.legend(title="Coverage fractions", fontsize=9, title_fontsize=10, ncol=2)
plt.tight_layout()
plt.show()
